# Cup of Coffee — free Colab GPU I2V proof

This notebook generates a **real 2.7-second image-to-video clip** with `Lightricks/LTX-Video` (2B). It never constructs motion from still-image transforms. FFmpeg is used only as the H.264 encoder after the neural video frames exist.

1. In Colab choose **Runtime → Change runtime type → T4 GPU**.
2. Choose **Runtime → Run all**.
3. Keep the tab open while the free runtime downloads the open weights and renders.

The notebook uses the real Cup of Coffee product/cafe photograph in `public/cup_of_coffee_HD_preserved.png`. The weekday-labelled lineup image is deliberately never loaded. Campaign typography and the official `public/logo.png` are composited only after generation.


In [ ]:
%pip install -q --upgrade "diffusers==0.35.2" "transformers>=4.49,<5" "accelerate>=1.2,<2" "bitsandbytes>=0.45" "safetensors>=0.4.5" "sentencepiece>=0.2" "imageio>=2.36" "imageio-ffmpeg>=0.6" "Pillow>=10.4"


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("A real CUDA runtime is required. In Colab select Runtime > Change runtime type > T4 GPU, then Run all again. No CPU or mock fallback is permitted.")

gpu_name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram_gib < 14.0:
    raise RuntimeError(f"{gpu_name} exposes only {vram_gib:.1f} GiB. This notebook requires a free-tier GPU with at least 14 GiB.")
print(json.dumps({"gpu": gpu_name, "vram_gib": round(vram_gib, 2), "torch": torch.__version__, "cuda": torch.version.cuda}, indent=2))

REPO = Path("/content/ai-video-studio")
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/alneval20/ai-video-studio.git", str(REPO)], check=True)
os.chdir(REPO)
OUTPUT_DIR = REPO / "outputs" / "amedspor-free-colab-ltx"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository: {REPO}")
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
from PIL import Image, ImageOps
from IPython.display import display

WIDTH = 576
HEIGHT = 1024
FPS = 24
NUM_FRAMES = 65  # LTX temporal rule: 8n+1; 65/24 = 2.708 seconds
SEED = 212026
MODEL_ID = "Lightricks/LTX-Video"

product_source = REPO / "public" / "cup_of_coffee_HD_preserved.png"
logo_source = REPO / "public" / "logo.png"
init_path = OUTPUT_DIR / "init-frame-576x1024.png"
for required in (product_source, logo_source):
    if not required.is_file():
        raise FileNotFoundError(required)

# Source pixels are stored sideways. This fixes orientation and prepares the
# model's single conditioning image; it does not generate or animate frames.
with Image.open(product_source) as source:
    upright = source.convert("RGB").transpose(Image.Transpose.ROTATE_270)
    init_image = ImageOps.fit(
        upright,
        (WIDTH, HEIGHT),
        method=Image.Resampling.LANCZOS,
        centering=(0.50, 0.50),
    )
init_image.save(init_path, quality=95)
assert init_image.size == (WIDTH, HEIGHT)
print(f"Conditioning image: {init_path}")
display(init_image.resize((288, 512)))


## Genuine neural I2V generation

This is the only motion-producing stage. LTX receives the real photograph as its init image and denoises 65 temporally coupled frames. No campaign copy is sent to the model, which keeps generated footage free of generated lettering.


In [ ]:
import gc
import time
from diffusers import AutoModel, LTXImageToVideoPipeline
from diffusers.hooks import apply_group_offloading
from transformers import BitsAndBytesConfig, T5EncoderModel

# T4 is a Turing GPU and has no native BF16 tensor cores. FP16 is the
# correct compute dtype there; newer free allocations may use BF16.
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Loading {MODEL_ID} with compute dtype {COMPUTE_DTYPE} ...")

# The bundled T5 encoder is 19 GB in full precision. Loading it in 8-bit
# keeps both the T4 VRAM and Colab Free host RAM inside their limits. It is
# used only to produce prompt embeddings, then unloaded before video denoising.
text_encoder = T5EncoderModel.from_pretrained(
    MODEL_ID,
    subfolder="text_encoder",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    torch_dtype=COMPUTE_DTYPE,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

transformer = AutoModel.from_pretrained(
    MODEL_ID,
    subfolder="transformer",
    torch_dtype=COMPUTE_DTYPE,
    low_cpu_mem_usage=True,
)
# Storage casting reduces resident weights. Layers are restored to the
# compute dtype before each matrix multiplication, so this is still the
# real model—not a mock or a frame transform.
transformer.enable_layerwise_casting(
    storage_dtype=torch.float8_e4m3fn,
    compute_dtype=COMPUTE_DTYPE,
)

pipe = LTXImageToVideoPipeline.from_pretrained(
    MODEL_ID,
    transformer=transformer,
    text_encoder=text_encoder,
    torch_dtype=COMPUTE_DTYPE,
    low_cpu_mem_usage=True,
)

onload_device = torch.device("cuda")
offload_device = torch.device("cpu")
pipe.transformer.enable_group_offload(
    onload_device=onload_device,
    offload_device=offload_device,
    offload_type="leaf_level",
    use_stream=False,
)
apply_group_offloading(
    pipe.vae,
    onload_device=onload_device,
    offload_device=offload_device,
    offload_type="leaf_level",
)
pipe.vae.enable_tiling()

PROMPT = (
    "A single continuous photorealistic premium beverage commercial inside the same Cup of Coffee cafe in the conditioning image. "
    "Preserve the exact iced drink, dessert, counter, cafe layout, product proportions, cup geometry and existing packaging marks. "
    "A restrained physical tabletop dolly-in with a very small three-quarter orbit creates real depth and foreground-to-background parallax. "
    "Transparent three-dimensional ice shows refraction, internal imperfections and wet surfaces; the coffee has subtle believable inertia. "
    "Condensation moves naturally down the cold cup while glass reflections and warm practical lighting change continuously with the camera. "
    "Subtle green and red match-day reflections remain environmental accents, never a stadium. Natural motion blur, shallow macro depth of field, stable product identity."
)
NEGATIVE_PROMPT = (
    "still image, frozen frame, slideshow, Ken Burns, 2D transform, animated poster, flat motion graphics, "
    "floating object, morphing cup, warped geometry, rubbery liquid, opaque plastic ice, impossible reflections, "
    "flicker, generated text, changing lettering, weekday labels, watermark, stadium, football pitch, sports poster"
)

with torch.inference_mode():
    (
        prompt_embeds,
        prompt_attention_mask,
        negative_prompt_embeds,
        negative_prompt_attention_mask,
    ) = pipe.encode_prompt(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        do_classifier_free_guidance=True,
        device=torch.device("cuda"),
        dtype=COMPUTE_DTYPE,
        max_sequence_length=128,
    )
pipe.text_encoder = None
del text_encoder
gc.collect()
torch.cuda.empty_cache()
print("Prompt embeddings encoded; 8-bit T5 unloaded before video denoising.")

generator = torch.Generator(device="cpu").manual_seed(SEED)
started = time.monotonic()
with torch.inference_mode():
    generated_frames = pipe(
        image=init_image,
        prompt=None,
        negative_prompt=None,
        prompt_embeds=prompt_embeds,
        prompt_attention_mask=prompt_attention_mask,
        negative_prompt_embeds=negative_prompt_embeds,
        negative_prompt_attention_mask=negative_prompt_attention_mask,
        width=WIDTH,
        height=HEIGHT,
        num_frames=NUM_FRAMES,
        frame_rate=FPS,
        num_inference_steps=30,
        guidance_scale=5.0,
        decode_timestep=0.05,
        decode_noise_scale=0.025,
        generator=generator,
        output_type="pil",
    ).frames[0]
elapsed = time.monotonic() - started
assert len(generated_frames) == NUM_FRAMES
print(f"Generated {len(generated_frames)} neural video frames in {elapsed / 60:.1f} minutes.")
del pipe, transformer
gc.collect()
torch.cuda.empty_cache()


In [ ]:
import imageio.v2 as imageio
import numpy as np

RAW_PATH = OUTPUT_DIR / "raw-ltx-2b-i2v-576x1024.mp4"
FINAL_PATH = OUTPUT_DIR / "composited-ltx-2b-i2v-576x1024.mp4"
MANIFEST_PATH = OUTPUT_DIR / "render-manifest.json"
DOWNLOAD_PATH = Path("/content/amedspor-free-real-i2v-test.mp4")

def write_h264(frames, path):
    writer = imageio.get_writer(
        str(path),
        format="FFMPEG",
        mode="I",
        fps=FPS,
        codec="libx264",
        pixelformat="yuv420p",
        quality=8,
        macro_block_size=None,
        ffmpeg_log_level="error",
    )
    try:
        for frame in frames:
            writer.append_data(np.asarray(frame.convert("RGB")))
    finally:
        writer.close()

write_h264(generated_frames, RAW_PATH)

# Reject an effectively frozen result. This is a signal check in addition
# to the stronger provenance guarantee: every frame came from LTX inference.
small = np.stack([
    np.asarray(frame.resize((72, 128)).convert("RGB"), dtype=np.float32)
    for frame in generated_frames
])
mean_adjacent_delta = float(np.abs(np.diff(small, axis=0)).mean())
first_last_delta = float(np.abs(small[-1] - small[0]).mean())
if mean_adjacent_delta < 0.10 or first_last_delta < 0.50:
    raise RuntimeError(
        f"Generated clip is effectively frozen (adjacent={mean_adjacent_delta:.3f}, first/last={first_last_delta:.3f}); refusing to present it as an I2V proof."
    )
print({"mean_adjacent_delta": mean_adjacent_delta, "first_last_delta": first_last_delta})


## Clean post-production only

The neural footage already exists at this point. This cell adds only the official logo and exact campaign copy as a static broadcast-safe overlay. It does not create camera or subject motion.


In [ ]:
from PIL import ImageDraw, ImageFont
import shutil

font_regular_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
font_bold_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
font_regular = ImageFont.truetype(font_regular_path, 25)
font_bold = ImageFont.truetype(font_bold_path, 32)
font_percent = ImageFont.truetype(font_bold_path, 108)
font_discount = ImageFont.truetype(font_bold_path, 34)

with Image.open(logo_source) as logo_file:
    official_logo = logo_file.convert("RGBA")
official_logo.thumbnail((112, 112), Image.Resampling.LANCZOS)

def composite_campaign(frame):
    base = frame.convert("RGBA")
    overlay = Image.new("RGBA", base.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)

    # Logo is only uniformly resized; aspect ratio and pixels are preserved.
    draw.rounded_rectangle((24, 24, 160, 160), radius=24, fill=(255, 255, 255, 224))
    overlay.alpha_composite(official_logo, (36, 36))

    draw.rounded_rectangle((20, 695, 556, 1004), radius=30, fill=(7, 10, 9, 184))
    draw.rectangle((20, 695, 286, 700), fill=(26, 167, 92, 255))
    draw.rectangle((286, 695, 556, 700), fill=(184, 34, 48, 255))
    draw.text((42, 720), "Amedspor’un maçlarının", font=font_regular, fill=(255, 255, 255, 255))
    draw.text((42, 754), "olduğu günlerde", font=font_bold, fill=(255, 255, 255, 255))
    draw.text((42, 803), "Tüm ürünlerde", font=font_discount, fill=(255, 255, 255, 255))
    draw.text((36, 835), "%21", font=font_percent, fill=(255, 255, 255, 255), stroke_width=2, stroke_fill=(11, 11, 11, 220))
    draw.text((316, 910), "indirim", font=font_discount, fill=(255, 255, 255, 255))
    return Image.alpha_composite(base, overlay).convert("RGB")

composited_frames = [composite_campaign(frame) for frame in generated_frames]
write_h264(composited_frames, FINAL_PATH)
shutil.copy2(FINAL_PATH, DOWNLOAD_PATH)

manifest = {
    "real_generation": True,
    "provider": "google-colab-free-cuda",
    "model_id": MODEL_ID,
    "pipeline": "LTXImageToVideoPipeline",
    "seed": SEED,
    "width": WIDTH,
    "height": HEIGHT,
    "fps": FPS,
    "num_frames": NUM_FRAMES,
    "duration_sec": NUM_FRAMES / FPS,
    "num_inference_steps": 30,
    "init_frame": str(init_path),
    "raw_mp4": str(RAW_PATH),
    "final_mp4": str(FINAL_PATH),
    "mean_adjacent_delta": mean_adjacent_delta,
    "first_last_delta": first_last_delta,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))


In [ ]:
from IPython.display import Video, display
from google.colab import files

probe = subprocess.run(
    [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=codec_name,width,height,r_frame_rate,nb_frames:format=duration",
        "-of", "json",
        str(FINAL_PATH),
    ],
    check=True,
    capture_output=True,
    text=True,
)
media = json.loads(probe.stdout)
stream = media["streams"][0]
assert stream["codec_name"] == "h264"
assert int(stream["width"]) == WIDTH and int(stream["height"]) == HEIGHT
print(json.dumps(media, indent=2))
print(f"FINAL MP4: {FINAL_PATH}")
display(Video(str(FINAL_PATH), embed=True, width=360))
files.download(str(DOWNLOAD_PATH))
